# Crop price index at the grid-cell level

> To write (Andrew): what this notebook does and why. Points to cover: we follow Berman et al. (2023),
> equations (1) and (2); each 5 arc-minute cell (about 9 km) gets its own price index, a weighted
> average of world crop prices where the weights are the relative agronomic suitability of the cell
> for each crop; suitability depends only on climate, soil and terrain, so it is not chosen by farmers.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio

plt.style.use("default")
plt.rcParams["figure.facecolor"] = "white"

# Data is read from GitHub so the notebook runs on any computer.
# Switch BRANCH to "main" once the branch is merged.
BRANCH = "andrew-cleanup-berman"
BASE = f"https://raw.githubusercontent.com/AndrewEberhardt/DataScienceProject/{BRANCH}/data/"
GAEZ_URL = BASE + "raw/gaez/sxHr_{}.tif"

# Crop -> (GAEZ v4 codes, World Bank Pink Sheet series). Rice: max of dryland and wetland rice.
# Coffee: average of the Arabica and Robusta prices.
CROPS = {
    "banana":    (["ban"], ["Banana, US"]),
    "cocoa":     (["coc"], ["Cocoa"]),
    "coconut":   (["con"], ["Coconut oil"]),
    "coffee":    (["cof"], ["Coffee, Arabica", "Coffee, Robusta"]),
    "cotton":    (["cot"], ["Cotton, A Index"]),
    "maize":     (["mze"], ["Maize"]),
    "oil palm":  (["olp"], ["Palm oil"]),
    "rice":      (["rcd", "rcw"], ["Rice, Thai 5%"]),
    "soybean":   (["soy"], ["Soybeans"]),
    "sugarcane": (["suc"], ["Sugar, world"]),
    "tea":       (["tea"], ["Tea, avg 3 auctions"]),
    "tobacco":   (["tob"], ["Tobacco, US import u.v."]),
    "wheat":     (["whe"], ["Wheat, US HRW"]),
    "barley":    (["brl"], ["Barley"]),
    "sorghum":   (["srg"], ["Sorghum"]),
}
CROPS_13 = list(CROPS)[:13]          # main index, 2001-2025
CROPS_15 = list(CROPS)               # Berman et al. crops, replication 2001-2018
YEARS_13 = list(range(2001, 2026))
YEARS_15 = list(range(2001, 2019))
SOURCE_GAEZ = "Source: FAO/IIASA GAEZ v4 (high input, rainfed, 1961-1990), Hansen et al. GFC v1.13."
SOURCE_PRICE = "Source: World Bank Commodity Price Data (Pink Sheet), real prices."

## 2. Grid cells

One row per GAEZ 5 arc-minute cell of the 62 sample countries with tree cover above 25 percent in 2000
(built from the Hansen pixels by `scripts/build_grid_panel.py` and `scripts/assemble_grid_panel.py`).

In [ ]:
cells = pd.read_csv(BASE + "processed/grid_cells.csv.gz",
                    usecols=["cell_id", "row", "col", "lat", "lon", "iso3", "in_tropics",
                             "pixel_area_ha", "forest25_px"])
regions = pd.read_csv(BASE + "processed/shock_panel.csv")[["iso3", "country", "region"]].drop_duplicates()
cells = cells.merge(regions, on="iso3", how="left")
cells["forest25_ha"] = cells.forest25_px * cells.pixel_area_ha
print(f"cells: {len(cells):,} | countries: {cells.iso3.nunique()} | in tropics band: {cells.in_tropics.sum():,}")
cells.groupby("region").agg(cells=("cell_id", "size"), forest_2000_Mha=("forest25_ha", lambda x: x.sum() / 1e6)).round(1)

## 3. Crop suitability

GAEZ suitability index (0 to 10,000) of each crop in each cell, rescaled to 0-1.
Water and missing cells (value -9) count as not suitable.

In [ ]:
suit = pd.DataFrame({"cell_id": cells.cell_id})
for crop, (codes, _) in CROPS.items():
    layers = []
    for c in codes:
        with rasterio.open(GAEZ_URL.format(c)) as src:
            assert src.shape == (2160, 4320) and src.transform.c == -180 and src.transform.f == 90
            a = src.read(1).astype(np.float32)
        layers.append(np.clip(a, 0, None)[cells.row.values, cells.col.values] / 10000)
    suit[crop] = np.max(layers, axis=0)

desc = suit[CROPS_15].describe().T[["mean", "50%", "max"]]
desc["share_cells_suitable"] = (suit[CROPS_15] >= 0.4).mean()   # GAEZ class medium or better
desc.round(3).sort_values("mean", ascending=False)

Cells with zero suitability for every crop have no price index (Berman et al. keep only cells with
suitability data). How much forest do we lose?

In [ ]:
for name, crops in [("13 crops", CROPS_13), ("15 crops", CROPS_15)]:
    none = suit[crops].sum(axis=1) == 0
    print(f"{name}: cells with no suitable crop {none.sum():,} ({none.mean():.1%}), "
          f"share of forest 2000 {cells.forest25_ha[none].sum() / cells.forest25_ha.sum():.1%}")
    print("   by region:", cells[none].region.value_counts().to_dict())

Share of the variance of suitability that is **within** countries (Berman et al., Table OA.1).
Country x year fixed effects remove everything national, so the index must vary inside countries.

In [ ]:
within = {}
for crop in CROPS_15:
    total = suit[crop].var()
    between = suit[crop].groupby(cells.iso3).transform("mean").var()
    within[crop] = 1 - between / total
pd.Series(within, name="share_within_country").sort_values().round(2).to_frame().T

## 4. World crop prices (base 100 in 2000)

In [ ]:
prices = pd.read_csv(BASE + "processed/commodity_prices_annual.csv")
P = {}
for crop, (_, series) in CROPS.items():
    s = (prices[prices.commodity.isin(series)]
         .groupby("year").price_real.mean())          # average when two series (coffee)
    P[crop] = 100 * s / s.loc[2000]
P = pd.DataFrame(P).loc[2000:2025]
print("last year with a price:", P.apply(lambda s: s.last_valid_index()).to_dict())
P.loc[[2000, 2008, 2011, 2018, 2022, 2025], CROPS_15].round(0)

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 8), sharex=True, sharey=True)
for ax, crop in zip(axes.flat, CROPS_15):
    ax.plot(P.index, P[crop], color="tab:green")
    ax.axhline(100, color="grey", lw=0.6)
    ax.set_title(crop)
for ax in axes[:, 0]:
    ax.set_ylabel("Real price (2000 = 100)")
for ax in axes[-1]:
    ax.set_xlabel("Year")
fig.suptitle("World crop prices used in the index, 2000-2025", fontsize=14)
fig.text(0.01, -0.01, SOURCE_PRICE + " Barley and sorghum stop in 2020.", fontsize=9)
fig.tight_layout()
fig.savefig("../figures/crop_prices.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Correlation of log prices across crops, 2001-2025 (Berman et al., Table OA.2)
corr = np.log(P.loc[2001:2025, CROPS_13]).corr()
print(f"pairwise correlations: min {corr.values[np.triu_indices(13, 1)].min():.2f}, "
      f"median {np.median(corr.values[np.triu_indices(13, 1)]):.2f}, max {corr.values[np.triu_indices(13, 1)].max():.2f}")
corr.round(2)

## 5. Relative suitability and price index

Berman et al. (2023), equations (1) and (2):
`alpha[i,c] = S[i,c] / sum_j S[j,c]` and `Price[c,t] = sum_i alpha[i,c] * P[i,t]`.

In [ ]:
def price_index(crops, years):
    """Log price index (cells x years) and the relative suitability weights."""
    S = suit[crops].to_numpy()
    ok = S.sum(axis=1) > 0
    alpha = np.full_like(S, np.nan)
    alpha[ok] = S[ok] / S[ok].sum(axis=1, keepdims=True)
    index = alpha @ P.loc[years, crops].to_numpy().T          # cells x years
    return pd.DataFrame(np.log(index), index=cells.cell_id, columns=years), pd.DataFrame(alpha, columns=crops)

lnP13, alpha13 = price_index(CROPS_13, YEARS_13)
lnP15, alpha15 = price_index(CROPS_15, YEARS_15)
print("13 crops:", lnP13.shape, "| 15 crops:", lnP15.shape)
lnP13.describe().T[["mean", "std", "min", "max"]].iloc[::4].round(3)

In [ ]:
# Mean weight of each crop over forest cells: which crops drive the index?
w = cells.forest25_ha / cells.forest25_ha.sum()
pd.DataFrame({
    "mean alpha (cells)": alpha13.mean(),
    "mean alpha (forest weighted)": alpha13.mul(w.values, axis=0).sum() / w[alpha13.notna().all(axis=1).values].sum(),
}).sort_values("mean alpha (forest weighted)", ascending=False).round(3)

## 6. Maps

In [ ]:
def to_grid(values):
    """Put one value per cell back on the 2160 x 4320 GAEZ grid, cropped to the sample."""
    g = np.full((2160, 4320), np.nan)
    g[cells.row.values, cells.col.values] = values
    r0, r1 = cells.row.min(), cells.row.max() + 1
    c0, c1 = cells.col.min(), cells.col.max() + 1
    extent = [-180 + c0 / 12, -180 + c1 / 12, 90 - r1 / 12, 90 - r0 / 12]
    return g[r0:r1, c0:c1], extent

dominant = alpha13.idxmax(axis=1)
codes = {c: k for k, c in enumerate(CROPS_13)}
g, extent = to_grid(dominant.map(codes).astype(float).values)
cmap = mcolors.ListedColormap(plt.cm.tab20.colors[:13])

fig, ax = plt.subplots(figsize=(15, 6))
ax.imshow(g, extent=extent, cmap=cmap, vmin=-0.5, vmax=12.5, interpolation="nearest")
handles = [plt.Rectangle((0, 0), 1, 1, color=cmap(k)) for k in range(13)]
ax.legend(handles, CROPS_13, ncol=7, loc="lower left", fontsize=8, title="Crop with the highest relative suitability")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Dominant crop in the price index of each forested 9 km cell")
fig.text(0.01, 0.01, SOURCE_GAEZ, fontsize=9)
fig.savefig("../figures/dominant_crop_map.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 11))
for ax, (lnP, y1, n) in zip(axes, [(lnP15, 2018, 15), (lnP13, 2025, 13)]):
    change = (lnP[y1] - lnP[2001]).values
    g, extent = to_grid(change)
    lim = np.nanpercentile(np.abs(change), 98)
    im = ax.imshow(g, extent=extent, cmap="RdBu_r", vmin=-lim, vmax=lim, interpolation="nearest")
    fig.colorbar(im, ax=ax, shrink=0.8, label="Change in log price index")
    ax.set_title(f"Change of the crop price index, 2001-{y1} ({n} crops)")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
fig.text(0.01, 0.0, SOURCE_GAEZ + " " + SOURCE_PRICE, fontsize=9)
fig.tight_layout()
fig.savefig("../figures/price_index_change_map.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. Identifying variation

The regression uses cell fixed effects and country x year fixed effects. What is left of the price index
after removing them is the variation that identifies beta.

In [ ]:
def residual_sd(lnP, n_iter=30):
    """SD of the log index raw, after cell FE, and after cell + country x year FE (alternating projections)."""
    long = lnP.reset_index().melt(id_vars="cell_id", var_name="year", value_name="y").dropna()
    long["iso3"] = long.cell_id.map(cells.set_index("cell_id").iso3)
    long["ct"] = long.iso3 + "_" + long.year.astype(str)
    y = long.y.to_numpy(np.float64).copy()
    out = {"raw": y.std()}
    y -= long.groupby("cell_id").y.transform("mean").to_numpy()
    out["after cell FE"] = y.std()
    for _ in range(n_iter):
        y -= pd.Series(y).groupby(long.ct.values).transform("mean").to_numpy()
        y -= pd.Series(y).groupby(long.cell_id.values).transform("mean").to_numpy()
    out["after cell + country x year FE"] = y.std()
    return pd.Series(out)

pd.DataFrame({"13 crops, 2001-2025": residual_sd(lnP13), "15 crops, 2001-2018": residual_sd(lnP15)}).round(4)

In [ ]:
# Average log index by region (forest weighted), relative to 2001
fig, ax = plt.subplots(figsize=(9, 5))
for region, grp in cells.groupby("region"):
    m = lnP13.loc[grp.cell_id].mul(grp.forest25_ha.values, axis=0).sum() / grp.forest25_ha.sum()
    ax.plot(m.index, m - m.loc[2001], label=region)
ax.axhline(0, color="grey", lw=0.6)
ax.set_xlabel("Year")
ax.set_ylabel("Log price index, change since 2001")
ax.set_title("Crop price index by region (13 crops, forest-weighted mean)")
ax.legend()
fig.text(0.01, -0.03, SOURCE_GAEZ + " " + SOURCE_PRICE, fontsize=9)
fig.savefig("../figures/price_index_by_region.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Save the suitability of each cell

The regression notebook rebuilds the index from this file (suitability index 0-10,000, 15 crops),
so both versions of the index (13 and 15 crops) come from the same numbers.

In [ ]:
out = suit.copy()
out[CROPS_15] = (out[CROPS_15] * 10000).round().astype("int16")
out.columns = ["cell_id"] + [f"suit_{c.replace(' ', '_')}" for c in CROPS_15]
out.to_csv("../data/processed/grid_suitability.csv.gz", index=False, compression="gzip")
print(out.shape)
out.head()